In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

df=pd.read_csv('dataset_for_dano_fuel.csv')

df['week'] = pd.to_datetime(df['week']).dt.tz_convert(None)
df = df.sort_values('week')

df['week_start'] = df['week'].dt.normalize() + pd.to_timedelta(1, 'D')
df['week_end']   = df['week_start'] + pd.to_timedelta(6, 'D')
df['week']   = (df['week'] - df['week'].min()).dt.days // 7 + 1 + 13  # приводим к номерам недель года

bad_parties = df[((df['week'] == 14) | (df['week'] >= 19)) & (df['service_fee_amt'] == 69)]['party_rk']
df = df[~df['party_rk'].isin(bad_parties)]

In [ ]:
df_stable = df[(df['week'] >= 19) & (df['week'] <= 22)].copy()

def assign_next_order_exact(sub_df):
    sub_df = sub_df.sort_values('week')
    weeks = sub_df['week'].values
    user_weeks = set(weeks)

    next_order_list = []

    for w in weeks:
        next_week = w + 1
        next_order = 1 if next_week in user_weeks else 0
        next_order_list.append(next_order)

    sub_df = sub_df.copy()
    sub_df['next_order'] = next_order_list
    return sub_df

df_next = df_stable.groupby('party_rk').apply(assign_next_order_exact).reset_index(drop=True)

print(df_next[['party_rk', 'week', 'service_fee_amt', 'next_order']].head(10))


In [ ]:
# --- 1. Группируем по уровню service_fee_amt ---
agg = df_next.groupby('service_fee_amt')['next_order'].mean().reset_index()
print(agg)

# --- 2. Фирменные цвета Т‑Банка ---
colors = [
    '#ffd600', '#f3c400', '#d4a700', '#b08500', '#8c6a00',
    '#a0a0a0', '#787878', '#505a65', '#2f3a45', '#1d1d1d', '#000000'
]

# --- 3. График ---
plt.figure(figsize=(15,3))
plt.plot(agg['service_fee_amt'], agg['next_order'], marker='o', color=colors[1], linewidth=2)

# --- 4. Гипотетический разрыв ---
plt.plot([29, 39], [agg.loc[agg['service_fee_amt']==29, 'next_order'].values[0],
                    agg.loc[agg['service_fee_amt']==39, 'next_order'].values[0]],
         color='red', linewidth=2, linestyle='--')

# --- 5. Подписи и оформление ---
plt.xlabel('Сервисный сбор')
plt.ylabel('Вероятность следующего заказа')
plt.title('Зависимость вероятности следующего заказа от сервисного сбора\n(гипотеза H1)')
plt.xticks(agg['service_fee_amt'])
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
import statsmodels.api as sm
import pandas as pd

# Категория
df_next['fee_cat'] = df_next['service_fee_amt'].astype('category')

# X = fee_cat в виде дамми-переменных
X = pd.get_dummies(df_next['fee_cat'], drop_first=True).astype(float)

# y = next_order как числовая переменная
y = df_next['next_order'].astype(int)

# Логит-модель
model = sm.Logit(y, X).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_next['party_rk']}
)
print(model.summary())

In [ ]:
import statsmodels.api as sm
import pandas as pd

# Категория
df_next['fee_cat'] = df_next['service_fee_amt'].astype('category')

# X = fee_cat в виде дамми-переменных
X = pd.get_dummies(df_next['fee_cat'], drop_first=True).astype(float)

# y = next_order как числовая переменная
y = df_next['next_order'].astype(int)

# Логит-модель
X = sm.add_constant(X)
model = sm.Logit(y, X).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_next['party_rk']}
)
print(model.summary())

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# Убедимся, что переменные корректного типа
df_next['next_order'] = df_next['next_order'].astype(int)
df_next['fee_cat'] = df_next['fee_cat'].astype('category')
df_next['platform'] = df_next['platform'].astype('category')

# Модель с контролями
formula = 'next_order ~ C(fee_cat) + cashback_category + np.log1p(gmv) + orders_cnt + entries_cnt + C(platform)'
model_full = smf.logit(formula, data=df_next).fit()
print(model_full.summary())

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Сравниваем 29 и 39
group_29 = df_next[df_next['service_fee_amt'] == 29]
group_39 = df_next[df_next['service_fee_amt'] == 39]

success_counts = [group_29['next_order'].sum(), group_39['next_order'].sum()]
nobs = [len(group_29), len(group_39)]

z_stat, p_value = proportions_ztest(success_counts, nobs)
print(f"Z-statistic: {z_stat:.4f}, p-value: {p_value:.4f}")

In [ ]:
# Создаем переменную "выше порога"
df_next['above_threshold'] = (df_next['service_fee_amt'] >= 39).astype(int)

# RDD модель
X_rdd = df_next[['above_threshold']]
X_rdd = sm.add_constant(X_rdd)
y = df_next['next_order']

model_rdd = sm.Logit(y, X_rdd).fit()
print(model_rdd.summary())

### Сводка результатов:
1. Z-тест между 29 и 39 рублями:
Z-statistic: 8.9579, p-value: 0.0000

Вывод: Разница статистически значима на любом разумном уровне значимости (p < 0.0001)

2. Логистическая регрессия без контроля:
Все коэффициенты значимы (p < 0.001)

Эффект от 39 рублей (-0.7117) значительно сильнее, чем от других уровней

3. Модель с контролями (расширенная):
R-квадрат улучшился с -0.0046 до 0.011

Ключевое наблюдение: После контроля за поведенческими факторами (gmv, orders_cnt, platform):

19 рублей становится положительно значимым (+0.0996, p < 0.001)

23 и 29 рублей становятся незначимыми (p > 0.05)

39 рублей остается сильно отрицательным (-0.3619, p < 0.001)

4. Пороговая модель (above_threshold):
Простая и эффективная модель

Порог ≥39 рублей снижает логарифм шансов на -0.4679 (p < 0.001)

Ключевые выводы:
1. Гипотеза H1 подтверждается:
Существует статистически значимый разрыв при переходе от 29 к 39 рублям

39 рублей - критический порог, после которого вероятность возврата резко падает

2. Эффект 39 рублей устойчив:
Остается статистически значимым во всех моделях

Эффект сохраняется после контроля за поведенческими факторами

Отношение шансов (OR) ≈ 0.49-0.70 (в зависимости от модели) → шансы возврата снижаются на 30-51%

3. Интересный нюанс с 19 рублями:
После контроля за поведением 19 рублей показывает положительный эффект (+0.0996)

Это может означать:

Эффект "якоря" - небольшой сбор делает сервис воспринимаемым как более ценный

Фильтрация нелояльных пользователей - те, кто готов платить небольшой сбор, более лояльны

Смещение в базовой модели из-за разных профилей пользователей

4. Влияние контрольных переменных:
orders_cnt: Сильный положительный эффект (+0.3200) → активные пользователи чаще возвращаются

log(gmv): Отрицательный эффект (-0.0908) → пользователи с большими чеками реже возвращаются (возможно, делают более редкие, но крупные покупки)

Platform: iOS-пользователи более лояльны, чем Android

Практические рекомендации для бизнеса:
1. Немедленные действия:
Избегать сервисного сбора 39 рублей - он вызывает чрезмерный отток

Рассмотреть установление "красной линии" на уровне 29 рублей

2. Оптимальная стратегия ценообразования:
0-29 рублей: Относительно безопасная зона (особенно 19-23 рубля)

19 рублей: Может быть оптимальным (показывает даже положительный эффект после контроля)

39 рублей: Критически опасный уровень


Научные выводы:
1. Теоретическая значимость:
Подтверждена концепция "точки разрыва" (breaking point) в reservation price

Обнаружен нелинейный эффект сервисного сбора на удержание

39 рублей выступает как коллективная граница reservation price для значительной части пользователей

2. Методологический вклад:
Retention N+1 доказала свою эффективность как метрика

Контроль за поведенческими переменными меняет интерпретацию эффектов

Пороговая модель показала хорошую объясняющую способность

3. Экономическая интерпретация:
Существует допустимый диапазон сервисных сборов (0-29 рублей)

39 рублей выходит за пределы willingness-to-pay для многих пользователей

Небольшие сборы (19 рублей) могут даже улучшать восприятие ценности

Заключение:
Гипотеза H1 полностью подтверждается - существует резкая граница на уровне 39 рублей, после которой вероятность оттока значительно увеличивается.

Рекомендуемое действие для бизнеса: Установить максимальный сервисный сбор на уровне 29 рублей, а оптимальным уровнем считать 19-23 рубля. 39 рублей следует исключить из возможных значений как вызывающий неприемлемо высокий отток.

Исследование демонстрирует, как даже на ограниченных данных можно выявить экономически значимые закономерности с помощью корректно подобранных метрик и статистических методов.

In [ ]:
# Код для визуализации
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))

# 1. Фактические вероятности
ax1.bar(['0', '19', '23', '29', '39'], 
        [0.423, 0.452, 0.432, 0.417, 0.329],
        color=['#2f3a45', '#2f3a45', '#2f3a45', '#2f3a45', '#ff6b6b'],
        alpha=0.8)
ax1.set_xlabel('Сервисный сбор')
ax1.set_ylabel('Вероятность возврата на след. неделе')
ax1.set_title('Retention N+1 по уровням сбора')
ax1.axvline(x=3.5, color='red', linestyle='--', alpha=0.5)
ax1.text(3.7, 0.35, 'Критический\nпорог', color='red')

# 2. Относительное изменение
changes = [-100, +6.8, -4.6, -3.6, -21.1]  # в процентах
ax2.bar(['0→19', '19→23', '23→29', '29→39'], 
        changes[1:], 
        color=['#2f3a45', '#2f3a45', '#2f3a45', '#ff6b6b'])
ax2.set_xlabel('Изменение сбора')
ax2.set_ylabel('Δ Retention, п.п.')
ax2.set_title('Изменение вероятности возврата')
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.5)

plt.tight_layout()

In [ ]:
# Визуализация доверительных интервалов
plt.figure(figsize=(8, 5))
groups = ['0', '19', '23', '29', '39']
means = [0.423, 0.452, 0.432, 0.417, 0.329]
errors = [0.002, 0.003, 0.005, 0.008, 0.007]  # примерные ошибки

plt.errorbar(groups, means, yerr=errors, fmt='o', 
             capsize=5, color='#2f3a45', linewidth=2)
plt.fill_between([3.5, 4.5], [0.25, 0.25], [0.45, 0.45], 
                 color='#ff6b6b', alpha=0.2)
plt.xlabel('Сервисный сбор')
plt.ylabel('Вероятность возврата')
plt.title('Retention N+1 с 95% доверительными интервалами')
plt.grid(True, alpha=0.3)

In [ ]:
# Визуализация: распределение пользователей по уровням сборов
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 5))

# 1. Состав пользователей по сбору
user_counts = df_next.groupby('service_fee_amt')['party_rk'].nunique()
ax1.bar(['0', '19', '23', '29', '39'], 
        user_counts.values,
        color=['#2f3a45']*5, alpha=0.7)
ax1.set_xlabel('Уровень сервисного сбора')
ax1.set_ylabel('Количество уникальных пользователей')
ax1.set_title('Распределение пользователей по группам')
ax1.text(0, user_counts[0]*1.02, f'{user_counts[0]:,}', 
         ha='center', fontsize=9)

# 2. Средние значения контрольных переменных
variables = ['gmv']
for i, var in enumerate(variables):
    means = df_next.groupby('service_fee_amt')[var].mean()
    ax2.plot(['0', '19', '23', '29', '39'], means.values, 
             marker='o', label=var, linewidth=2)
ax2.set_xlabel('Сервисный сбор')
ax2.set_ylabel('Среднее значение')
ax2.set_title('Различия в профилях пользователей')
ax2.legend()
ax2.grid(True, alpha=0.3)

variables = ['orders_cnt']
for i, var in enumerate(variables):
    means = df_next.groupby('service_fee_amt')[var].mean()
    ax3.plot(['0', '19', '23', '29', '39'], means.values, 
             marker='o', label=var, linewidth=2)
ax3.set_xlabel('Сервисный сбор')
ax3.set_ylabel('Среднее значение')
ax3.set_title('Различия в профилях пользователей')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_next['log_gmv'] = np.log1p(df_next['gmv'])
df_next['cashback_sqrt'] = np.sqrt(df_next['cashback_category'])

# Контрольные переменные
num_vars = ['log_gmv', 'orders_cnt', 'entries_cnt', 'cashback_category']

# Настройка figure с 2x2 подграфиками
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

n_bins = 1000  # количество бинов для усреднения

for i, var in enumerate(num_vars):
    # Разбиваем на квантили
    df_next['bin'] = pd.qcut(df_next[var], n_bins, duplicates='drop')
    grouped = df_next.groupby('bin')['next_order'].mean().reset_index()
    grouped['x_mean'] = df_next.groupby('bin')[var].mean().values
    grouped['logit'] = np.log(grouped['next_order'] / (1 - grouped['next_order']))

    ax = axes[i]
    ax.scatter(grouped['x_mean'], grouped['logit'], color='#f3c400', label='Среднее в бине')
    ax.plot(grouped['x_mean'], grouped['logit'], color='#b08500', linewidth=2)
    ax.set_xlabel(var)
    ax.set_ylabel('logit(next_order)')
    ax.set_title(f'Линейность логита: {var}')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = df_next.copy()
df = df[df['entries_cnt'] <= 14]

epsilon = 1e-5

def compute_logit(p):
    p = np.clip(p, epsilon, 1-epsilon)
    return np.log(p / (1-p))

# Создаем новые бины
df['cashback_bin'] = pd.cut(df['cashback_category'], bins=[-1, 9, df['cashback_category'].max()])
df['orders_bin'] = pd.cut(df['orders_cnt'], bins=[0, 4, df['orders_cnt'].max()])
df['gmv_bin'] = pd.cut(df['gmv'],
                            bins=[0, 5000, 15000, df['gmv'].max()+1],
                            labels=['0-5000', '5000-15000', '15000+'],
                            include_lowest=True)
df['entries_bin'] = pd.cut(
    df['entries_cnt'],
    bins=[0, 5, 10, df['entries_cnt'].max()],
    labels=['0-5', '5-10', '10+'],
    include_lowest=True
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, var in enumerate(['gmv_bin', 'orders_bin', 'entries_bin', 'cashback_bin']):
    grouped = df.groupby(var)['next_order'].mean().reset_index()
    # grouped['x_mean'] = df.groupby(var)[var].apply(lambda x: x.cat.codes.mean()).values
    grouped['x_mean'] = grouped.index.values
    grouped['logit'] = compute_logit(grouped['next_order'])
    
    ax = axes[i]
    ax.plot(grouped['x_mean'], grouped['logit'], marker='o', linestyle='-', color='#b08500')

    ax.set_xticks(grouped['x_mean'])
    ax.set_xticklabels(grouped[var].astype(str), rotation=0, ha='center')

    ax.set_xlabel(var)
    ax.set_ylabel('logit(next_order)')
    ax.set_title(f'Логит возврата vs {var}')
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# Убедимся, что переменные корректного типа
df_next['next_order'] = df_next['next_order'].astype(int)
df_next['fee_cat'] = df_next['fee_cat'].astype('category')
df_next['platform'] = df_next['platform'].astype('category')

df_next['cashback_bin'] = pd.cut(df_next['cashback_category'], bins=[-1, 9, df['cashback_category'].max()])
df_next['orders_bin'] = pd.cut(df_next['orders_cnt'], bins=[0, 4, df['orders_cnt'].max()])
df_next['gmv_bin'] = pd.cut(df_next['gmv'],
                            bins=[0, 5000, 15000, df_next['gmv'].max()+1],
                            labels=['0-5000', '5000-15000', '15000+'],
                            include_lowest=True)
df_next['entries_bin'] = pd.cut(
    df_next['entries_cnt'],
    bins=[0, 5, 10, df['entries_cnt'].max()],
    labels=['0-5', '5-10', '10+'],
    include_lowest=True
)
df_next['entries_bin'] = df_next['entries_bin'].cat.add_categories('missing')
df_next['entries_bin'] = df_next['entries_bin'].fillna('missing')
df_next

# Модель с контролями
formula = 'next_order ~ C(fee_cat) + C(cashback_bin) + C(gmv_bin)  + C(orders_bin) + C(entries_bin) + C(platform)'
model_full = smf.logit(formula, data=df_next).fit(
    cov_type='cluster', cov_kwds={'groups': df_next['party_rk']},  # используем логистическую регрессию с кластерными ошибками
)
print(model_full.summary())

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

interval_cols = ["cashback_bin", "gmv_bin", "orders_bin", "entries_bin"]
df_next[interval_cols] = df_next[interval_cols].astype(str)

# Формируем дизайн-матрицу (дамми для категорий)
X = pd.get_dummies(
    df_next[['fee_cat', 'cashback_bin', 'gmv_bin', 'orders_bin', 'entries_bin', 'platform']],
    drop_first=True
)
# Добавляем константу — VIF требует
X = X.astype(float)
X = sm.add_constant(X)

vif_df = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

vif_df